In [1]:
!python --version

Python 3.11.14


## GDC

https://docs.gdc.cancer.gov/API/Users_Guide/Downloading_Files/

sample  
 ├── primary_site       (organ)  
 ├── disease_type       (broad class)  
 ├── subtype_global     (cross-cancer)  
 ├── subtype_tissue     (within-cancer)  
 └── histology          (biological family)  

### TCGA / GDC Portal

https://portal.gdc.cancer.gov/

In [2]:
import os, sys
import pandas as pd

from pathlib import Path

ROOT = Path().resolve().parent.parent
SRC = os.path.join(ROOT, "src")

if str(SRC) not in sys.path:
    sys.path.append(str(SRC))

print("ROOT:", ROOT)
print("SRC added:", SRC)

from libs.tcga_gdc_lib import *
from libs.Basic import *


ROOT: /home/flavio/uv
SRC added: /home/flavio/uv/src


### Defaults

In [3]:
ROOT = Path().resolve().parent
root_data = os.path.join(ROOT, "data/tcga")

gdc = GDC(root_data=root_data)

os.listdir(root_data)


['cases_for_PS_TCGA-LUAD.tsv',
 'subtype_for_PS_TCGA-ACC.tsv',
 'subtype_for_PS_TCGA-HNSC.tsv',
 'subtype_for_PS_TCGA-KIRP.tsv',
 'subtype_for_PS_TCGA-OV.tsv',
 'stage_for_PS_TCGA-UCEC_Subtype_basaloid squamous cell carcinoma.tsv',
 'subtype_for_PS_TCGA-UCS.tsv',
 'subtype_for_PS_TCGA-PAAD.tsv',
 'stage_for_PS_TCGA-UCEC_Subtype_papillary serous cystadenocarcinoma.tsv',
 'cases_for_PS_TCGA-BRCA.tsv',
 'subtype_for_PS_TCGA-LIHC.tsv',
 'subtype_for_PS_TCGA-STAD.tsv',
 'subtype_for_PS_TCGA-KIRC.tsv',
 'subtype_for_PS_TCGA-PRAD.tsv',
 'subtype_for_PS_TCGA-BLCA.tsv',
 'subtype_for_PS_TCGA-THYM.tsv',
 'subtype_for_PS_TCGA-COAD.tsv',
 'subtype_for_PS_TCGA-GBM.tsv',
 'subtype_for_PS_TCGA-PCPG.tsv',
 'programs.txt',
 'subtype_for_PS_TCGA-SKCM.tsv',
 'subtype_for_PS_TCGA-KICH.tsv',
 'subtype_for_PS_TCGA-CESC.tsv',
 'primary_site_program_TCGA.tsv',
 'subtype_for_PS_TCGA-TGCT.tsv',
 'subtype_for_PS_TCGA-THCA.tsv',
 'subtype_for_PS_TCGA-UVM.tsv',
 'subtype_for_PS_TCGA-LAML.tsv',
 'subtype_for_PS_TCG

### Get GDC programs - get_gdc_progams()

> end_point: url_gdc_project = "https://api.gdc.cancer.gov/projects"  
> "facets": "program.name"  

### Get valid primary sites - get_primary_sites()

> end_point: url_gdc_project = "https://api.gdc.cancer.gov/projects"  
> "fields": "project_id,name,primary_site,disease_type"  

### Get valid subtypes - get_subtypes()

> end_point: url_gdc_cases = "https://api.gdc.cancer.gov/cases"  
> facets = "diagnoses.primary_diagnosis"  

### For each subtype → get stages  - get_stages()

> end_point: url_gdc_cases = "https://api.gdc.cancer.gov/cases"  
> "field": "diagnoses.ajcc_pathologic_stage" - AJCC diagnoses   

### For each (subtype, stage) → get_samples()

> end_point: url_gdc_cases = "https://api.gdc.cancer.gov/cases"  
> given pid, subtype, stage  
> from cases   
> "samples.sample_id", "samples.submitter_id", "samples.sample_type"  

### get RNA-seq files - get_expression_files_given_samples()

> given: "field": "cases.case_id", "value": case_ids  
> end_point: url_gdc_files = "https://api.gdc.cancer.gov/files"  



### All programs

In [4]:
force=False
verbose=True

prog_list = gdc.get_gdc_progams(force=force, verbose=verbose)


File read at '/home/flavio/uv/perturb_agent/data/tcga/programs.txt'


In [5]:
np.array(prog_list)

array(['TCGA', 'MATCH', 'TARGET', 'CGCI', 'CMI', 'APOLLO', 'BEATAML1.0',
       'CPTAC', 'MP2PRT', 'ALCHEMIST', 'CCDI', 'CCG', 'CDDP_EAGLE',
       'CTSP', 'EXCEPTIONAL_RESPONDERS', 'FM', 'HCMI', 'MMRF', 'NCICCR',
       'OHSU', 'ORGANOID', 'RC', 'REBC', 'TRIO', 'VAREPOP', 'WCDT'],
      dtype='<U22')

### Primary sites given a programa

In [6]:
gdc.url_gdc_project

'https://api.gdc.cancer.gov/projects'

In [7]:
force=False
verbose=True

program = 'TCGA'

dfc = gdc.get_primary_sites(program=program, force=force, verbose=verbose)

print(len(dfc))
dfc.head(12)


Table opened ((33, 5)) at '/home/flavio/uv/perturb_agent/data/tcga/primary_site_program_TCGA.tsv'
33


,pid,primary_site,project_id,disease_type,name
0,TCGA-ACC,Adrenal gland,TCGA-ACC,Adenomas and Adenocarcinomas,Adrenocortical Carcinoma
1,TCGA-PCPG,"Adrenal gland, Retroperitoneum and peritoneum,...",TCGA-PCPG,Paragangliomas and Glomus Tumors,Pheochromocytoma and Paraganglioma
2,TCGA-BLCA,Bladder,TCGA-BLCA,"Epithelial Neoplasms, NOS, Squamous Cell Neopl...",Bladder Urothelial Carcinoma
3,TCGA-LGG,Brain,TCGA-LGG,Gliomas,Brain Lower Grade Glioma
4,TCGA-GBM,Brain,TCGA-GBM,"Not Reported, Gliomas",Glioblastoma Multiforme
5,TCGA-BRCA,Breast,TCGA-BRCA,"Adnexal and Skin Appendage Neoplasms, Adenomas...",Breast Invasive Carcinoma
6,TCGA-LUAD,Bronchus and lung,TCGA-LUAD,"Ductal and Lobular Neoplasms, Cystic, Mucinous...",Lung Adenocarcinoma
7,TCGA-LUSC,Bronchus and lung,TCGA-LUSC,"Squamous Cell Neoplasms, Adenomas and Adenocar...",Lung Squamous Cell Carcinoma
8,TCGA-MESO,"Bronchus and lung, Heart, mediastinum, and pleura",TCGA-MESO,Mesothelial Neoplasms,Mesothelioma
9,TCGA-CESC,"Cervix uteri, Ovary",TCGA-CESC,"Squamous Cell Neoplasms, Cystic, Mucinous and ...",Cervical Squamous Cell Carcinoma and Endocervi...


In [8]:
dfc.tail(3)

,pid,primary_site,project_id,disease_type,name
30,TCGA-THCA,Thyroid gland,TCGA-THCA,"Squamous Cell Neoplasms, Epithelial Neoplasms,...",Thyroid Carcinoma
31,TCGA-UCS,"Uterus, NOS, Corpus uteri",TCGA-UCS,"Basal Cell Neoplasms, Complex Mixed and Stroma...",Uterine Carcinosarcoma
32,TCGA-UCEC,"Uterus, NOS, Corpus uteri",TCGA-UCEC,"Not Reported, Cystic, Mucinous and Serous Neop...",Uterine Corpus Endometrial Carcinoma


### Subtypes

👉 GDC does NOT give you clean biological subtypes
👉 You must derive them yourself

This is actually a feature, not a bug — because:

> you control granularity  
> you can harmonize across cancers  
> you avoid noisy labels  


💡 If you want to level this up

I can help you build:

🔬 A cross-TCGA subtype harmonizer
maps all projects → unified subtype ontology
handles synonyms (adenocarcinoma, NOS, etc.)
outputs clean ML-ready labels

👉 That would massively improve your perturbation analysis.

- Brain tumors do NOT use AJCC staging (3 - TCGA-LGG Brain)

In [16]:
force=True
verbose=True

i = 5
pid = dfc.iloc[i].pid
primary_site = dfc.iloc[i].primary_site

print(i, pid, primary_site)

df_cases, df_subt, df_prof = gdc.get_cases_and_subtypes(pid=pid, batch_size=200, do_filter=False, force=force, verbose=verbose)

print("df_cases", len(df_cases), "df_subt", len(df_subt))
df_subt

5 TCGA-BRCA Breast
.......

👉 Returned 1098 / Total paginated 1098
Table saved ((1098, 24)) at '/home/flavio/uv/perturb_agent/data/tcga/cases_for_PS_TCGA-BRCA.tsv'
Table saved ((12, 6)) at '/home/flavio/uv/perturb_agent/data/tcga/subtype_for_PS_TCGA-BRCA.tsv'
df_cases 1098 df_subt 12


,pid,subtype_global,tumor_class,subtype_tissue,sstage,n
0,TCGA-BRCA,other,other,other,II,447
1,TCGA-BRCA,lobular,other,breast_lobular,missing,223
2,TCGA-BRCA,other,other,other,III,154
3,TCGA-BRCA,other,other,other,I,141
4,TCGA-BRCA,other,other,other,missing,85
5,TCGA-BRCA,adenocarcinoma_generic,adenocarcinoma,adenocarcinoma_generic,missing,19
6,TCGA-BRCA,other,other,other,IV,16
7,TCGA-BRCA,ductal,adenocarcinoma,breast_ductal,missing,6
8,TCGA-BRCA,ductal,other,breast_ductal,II,2
9,TCGA-BRCA,clear_cell,other,clear_cell,II,2


In [17]:
cols = ["pid", "subtype_global", "tumor_class", "subtype_tissue", "stage"]

df_cases[cols].head(12)

,pid,subtype_global,tumor_class,subtype_tissue,stage
0,TCGA-BRCA,other,other,other,unknown
1,TCGA-BRCA,other,other,other,Stage IIIA
2,TCGA-BRCA,other,other,other,Stage IA
3,TCGA-BRCA,other,other,other,Stage IIB
4,TCGA-BRCA,other,other,other,Stage IA
5,TCGA-BRCA,other,other,other,Stage IIA
6,TCGA-BRCA,other,other,other,Stage IA
7,TCGA-BRCA,other,other,other,Stage IIIA
8,TCGA-BRCA,other,other,other,Stage IIA
9,TCGA-BRCA,lobular,other,breast_lobular,unknown


In [ ]:
cols = ["case_id", "subtype_global", "tumor_class", "subtype_tissue", "stage"]
df_cases[cols]

In [18]:
df_prof

,primary_diagnosis,stage,tumor_grade,diagnosis_conflict,n_diagnoses
0,"Metaplastic carcinoma, NOS",Stage IIB,None,False,1
1,"Infiltrating duct carcinoma, NOS",Stage IIIA,None,True,2
2,"Infiltrating duct carcinoma, NOS",Stage IA,None,False,1
3,"Infiltrating duct carcinoma, NOS",Stage IIB,None,False,1
4,"Infiltrating duct carcinoma, NOS",Stage IA,None,False,1
...,...,...,...,...,...
1093,"Infiltrating duct carcinoma, NOS",Stage IIA,None,True,2
1094,"Infiltrating duct carcinoma, NOS",Stage IIA,None,False,1
1095,"Infiltrating duct carcinoma, NOS",Stage IIB,None,False,1
1096,"Lobular carcinoma, NOS",Stage IIB,None,False,1


In [20]:
dfc.tail(3)

,pid,primary_site,project_id,disease_type,name
30,TCGA-THCA,Thyroid gland,TCGA-THCA,"Squamous Cell Neoplasms, Epithelial Neoplasms,...",Thyroid Carcinoma
31,TCGA-UCS,"Uterus, NOS, Corpus uteri",TCGA-UCS,"Basal Cell Neoplasms, Complex Mixed and Stroma...",Uterine Carcinosarcoma
32,TCGA-UCEC,"Uterus, NOS, Corpus uteri",TCGA-UCEC,"Not Reported, Cystic, Mucinous and Serous Neop...",Uterine Corpus Endometrial Carcinoma


In [21]:
force=False
verbose=True

for i in range(len(dfc)):
    print(i, end=' ')
    pid = dfc.iloc[i].pid
    primary_site = dfc.iloc[i].primary_site

    print(pid, primary_site)

    df_cases, df_subt, df_prof = gdc.get_cases_and_subtypes(pid=pid, batch_size=200, do_filter=False, force=force, verbose=verbose)



0 TCGA-ACC Adrenal gland
Table opened ((92, 24)) at '/home/flavio/uv/perturb_agent/data/tcga/cases_for_PS_TCGA-ACC.tsv'
1 TCGA-PCPG Adrenal gland, Retroperitoneum and peritoneum, Other endocrine glands and related structures, Other and ill-defined sites, Connective, subcutaneous and other soft tissues, Spinal cord, cranial nerves, and other parts of central nervous system, Heart, mediastinum, and pleura
Table opened ((179, 24)) at '/home/flavio/uv/perturb_agent/data/tcga/cases_for_PS_TCGA-PCPG.tsv'
2 TCGA-BLCA Bladder
Table opened ((412, 24)) at '/home/flavio/uv/perturb_agent/data/tcga/cases_for_PS_TCGA-BLCA.tsv'
3 TCGA-LGG Brain
Table opened ((516, 24)) at '/home/flavio/uv/perturb_agent/data/tcga/cases_for_PS_TCGA-LGG.tsv'
4 TCGA-GBM Brain
Table opened ((617, 24)) at '/home/flavio/uv/perturb_agent/data/tcga/cases_for_PS_TCGA-GBM.tsv'
5 TCGA-BRCA Breast
Table opened ((1098, 24)) at '/home/flavio/uv/perturb_agent/data/tcga/cases_for_PS_TCGA-BRCA.tsv'
6 TCGA-LUAD Bronchus and lung
Table 

### Given a PS, Subtype, and Stage --> return the SAMPLES

In [ ]:
force=False
verbose=True

print(pid, subtype, stage)

df_sample = gdc.get_samples(pid=pid, subtype=subtype, stage=stage, batch_size=200, force=force, verbose=verbose)
print(len(df_sample))
df_sample.head(8)

In [ ]:
force=False
verbose=True

case_ids = list(np.unique(df_sample.case_id))

print(pid, subtype, stage, f"cases {len(case_ids)}")

df_files = gdc.get_expression_files_given_samples(pid=pid, subtype=subtype, stage=stage, 
                                                  case_ids=case_ids, batch_size=20, force=force, verbose=verbose)
print(len(df_files))
df_files.head(6)

In [ ]:
case_ids[:10]